***FRECUENCIA 10 MINS***

In [ ]:
import requests

r = requests.get("https://api.binance.com/api/v3/exchangeInfo", timeout=30)

print("HTTP:", r.status_code)
print(r.text[:1000])

HTTP: 451
{
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}


## Price


In [ ]:
import io
import re
import threading
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

S3_URL = "https://s3-ap-northeast-1.amazonaws.com/data.binance.vision"
DATA_URL = "https://data.binance.vision"
ROOT = "data/spot/monthly/klines/"
START = pd.Timestamp("2026-01-01", tz="UTC")
END = pd.Timestamp("2026-07-01", tz="UTC")
MONTHS = {f"2026-{m:02d}" for m in range(1, 7)}
WORKERS = 16
TIMEOUT = (10, 90)
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

_local = threading.local()

def get_session():
    if not hasattr(_local, "session"):
        retry = Retry(
            total=5,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"])
        )
        session = requests.Session()
        session.mount(
            "https://",
            HTTPAdapter(max_retries=retry, pool_connections=WORKERS, pool_maxsize=WORKERS)
        )
        _local.session = session
    return _local.session

def list_symbol_prefixes():
    prefixes = []
    marker = None

    while True:
        params = {
            "prefix": ROOT,
            "delimiter": "/",
            "max-keys": 1000
        }

        if marker:
            params["marker"] = marker

        r = get_session().get(S3_URL, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        root = ET.fromstring(r.content)

        page = [
            x.text
            for x in root.findall("s3:CommonPrefixes/s3:Prefix", NS)
            if x.text
        ]

        prefixes.extend(page)

        truncated = root.findtext("s3:IsTruncated", "false", NS) == "true"

        if not truncated:
            break

        next_marker = root.findtext("s3:NextMarker", None, NS)

        if next_marker:
            marker = next_marker
        elif page:
            marker = page[-1]
        else:
            raise RuntimeError("No fue posible continuar la paginación S3")

    return prefixes

def get_symbols():
    return sorted({
        prefix[len(ROOT):].rstrip("/")
        for prefix in list_symbol_prefixes()
        if prefix.startswith(ROOT)
    })

def get_months(symbol):
    prefix = f"{ROOT}{symbol}/5m/{symbol}-5m-2026-"

    r = get_session().get(
        S3_URL,
        params={"prefix": prefix, "max-keys": 1000},
        timeout=TIMEOUT
    )
    r.raise_for_status()

    root = ET.fromstring(r.content)

    keys = [
        x.text
        for x in root.findall("s3:Contents/s3:Key", NS)
        if x.text
    ]

    pattern = re.compile(
        rf"^{re.escape(ROOT)}{re.escape(symbol)}/5m/"
        rf"{re.escape(symbol)}-5m-(2026-\d{{2}})\.zip$"
    )

    return sorted({
        match.group(1)
        for key in keys
        if (match := pattern.match(key)) and match.group(1) in MONTHS
    })

def load_month(symbol, month):
    url = f"{DATA_URL}/{ROOT}{symbol}/5m/{symbol}-5m-{month}.zip"

    r = get_session().get(url, timeout=TIMEOUT)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = next(name for name in z.namelist() if name.endswith(".csv"))

        data = pd.read_csv(
            z.open(name),
            header=None,
            usecols=[0, 4],
            names=["timestamp", "close"],
            dtype={"timestamp": "int64", "close": "float32"}
        )

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        unit="us",
        utc=True
    )

    return data.set_index("timestamp")["close"]

def load_symbol(symbol):
    months = get_months(symbol)

    if not months:
        return None

    data = pd.concat(
        [load_month(symbol, month) for month in months],
        copy=False
    )

    data = data[
        (~data.index.duplicated(keep="last")) &
        (data.index >= START) &
        (data.index < END)
    ].sort_index()

    data = data[data.index.minute % 10 == 5]

    if data.empty:
        return None

    data.index = data.index.floor("10min")

    return data.astype("float32").rename(symbol)

symbols = get_symbols()

print(f"Símbolos históricos encontrados: {len(symbols)}")

series = {}
failed = {}

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(load_symbol, symbol): symbol
        for symbol in symbols
    }

    for i, future in enumerate(as_completed(futures), 1):
        symbol = futures[future]

        try:
            result = future.result()

            if result is not None:
                series[symbol] = result

        except Exception as e:
            failed[symbol] = str(e)

        if i % 100 == 0 or i == len(futures):
            print(
                f"{i}/{len(futures)} | "
                f"válidos: {len(series)} | "
                f"errores: {len(failed)}"
            )

if not series:
    raise RuntimeError("No se encontraron datos Spot para H1 2026")

index = pd.date_range(
    START,
    END,
    freq="10min",
    inclusive="left"
)

df = pd.DataFrame(series, index=index, dtype="float32")
df = df.reindex(sorted(df.columns), axis=1)
df.index.name = "timestamp"

print(f"Dimensión final: {df.shape}")
print(f"Pares reales: {df.shape[1]}")
print(f"RAM: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Errores: {len(failed)}")

Símbolos históricos encontrados: 3695


100/3695 | válidos: 55 | errores: 0
200/3695 | válidos: 95 | errores: 0


300/3695 | válidos: 144 | errores: 0


400/3695 | válidos: 205 | errores: 0


500/3695 | válidos: 247 | errores: 0


...

2600/3695 | válidos: 1208 | errores: 0
2700/3695 | válidos: 1253 | errores: 0
2800/3695 | válidos: 1318 | errores: 0


2900/3695 | válidos: 1375 | errores: 0
3000/3695 | válidos: 1425 | errores: 0


3100/3695 | válidos: 1474 | errores: 0
3200/3695 | válidos: 1528 | errores: 0
3300/3695 | válidos: 1576 | errores: 0
3400/3695 | válidos: 1620 | errores: 0


3500/3695 | válidos: 1671 | errores: 0


3600/3695 | válidos: 1724 | errores: 0
3695/3695 | válidos: 1788 | errores: 0
Dimensión final: (26064, 1788)
Pares reales: 1788
RAM: 177.97 MB
Errores: 0


In [ ]:
df[["BTCARS","BTCUSDT", "SOLUSDT", "BNBUSDT", "TRUUSDT", "ETHUSDT"]].head(5)

,BTCARS,BTCUSDT,SOLUSDT,BNBUSDT,TRUUSDT,ETHUSDT
timestamp,,,,,,
2026-01-01 00:00:00+00:00,135020208.0,87764.062500,125.010002,865.099976,0.0098,2978.080078
2026-01-01 00:10:00+00:00,135020208.0,87760.187500,124.949997,865.719971,0.0099,2977.620117
2026-01-01 00:20:00+00:00,135206528.0,87776.179688,125.160004,866.289978,0.0099,2979.750000
2026-01-01 00:30:00+00:00,135218256.0,87748.101562,125.110001,866.049988,0.0098,2979.560059
2026-01-01 00:40:00+00:00,135483520.0,87813.171875,125.230003,866.070007,0.0099,2981.840088


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26064 entries, 2026-01-01 00:00:00+00:00 to 2026-06-30 23:50:00+00:00
Freq: 10min
Columns: 1788 entries, 0GBNB to 币安人生USDT
dtypes: float32(1788)
memory usage: 178.0 MB


In [ ]:
columnas = pd.DataFrame({"columna": df.columns})
columnas

,columna
0,0GBNB
1,0GFDUSD
2,0GTRY
3,0GUSDC
4,0GUSDT
...,...
1783,币安人生TRY
1784,币安人生U
1785,币安人生USD1
1786,币安人生USDC


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=columnas)

https://docs.google.com/spreadsheets/d/1ukGVKUQqaI76MnsxXyJsGrVxkefcpshyidXoYNpviUc/edit#gid=0


In [ ]:
nan_por_columna = df.isna().sum().rename("NaNs").to_frame()
nan_por_columna = nan_por_columna[nan_por_columna["NaNs"] > 0]
nan_por_columna

,NaNs
0GBNB,23280
0GFDUSD,21840
1000SATSFDUSD,24894
1INCHBTC,9774
1MBABYDOGEFDUSD,23280
...,...
币安人生TRY,14754
币安人生U,15888
币安人生USD1,15888
币安人生USDC,948


## Volume

In [ ]:
import io
import re
import threading
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

S3_URL = "https://s3-ap-northeast-1.amazonaws.com/data.binance.vision"
DATA_URL = "https://data.binance.vision"
ROOT = "data/spot/monthly/klines/"
START = pd.Timestamp("2026-01-01", tz="UTC")
END = pd.Timestamp("2026-07-01", tz="UTC")
MONTHS = {f"2026-{m:02d}" for m in range(1, 7)}
WORKERS = 16
TIMEOUT = (10, 90)
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

_local = threading.local()

def get_session():
    if not hasattr(_local, "session"):
        retry = Retry(
            total=5,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"])
        )
        session = requests.Session()
        session.mount(
            "https://",
            HTTPAdapter(max_retries=retry, pool_connections=WORKERS, pool_maxsize=WORKERS)
        )
        _local.session = session
    return _local.session

def list_symbol_prefixes():
    prefixes = []
    marker = None

    while True:
        params = {
            "prefix": ROOT,
            "delimiter": "/",
            "max-keys": 1000
        }

        if marker:
            params["marker"] = marker

        r = get_session().get(S3_URL, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        root = ET.fromstring(r.content)

        page = [
            x.text
            for x in root.findall("s3:CommonPrefixes/s3:Prefix", NS)
            if x.text
        ]

        prefixes.extend(page)

        truncated = root.findtext("s3:IsTruncated", "false", NS) == "true"

        if not truncated:
            break

        next_marker = root.findtext("s3:NextMarker", None, NS)

        if next_marker:
            marker = next_marker
        elif page:
            marker = page[-1]
        else:
            raise RuntimeError("No fue posible continuar la paginación S3")

    return prefixes

def get_symbols():
    return sorted({
        prefix[len(ROOT):].rstrip("/")
        for prefix in list_symbol_prefixes()
        if prefix.startswith(ROOT)
    })

def get_months(symbol):
    prefix = f"{ROOT}{symbol}/5m/{symbol}-5m-2026-"

    r = get_session().get(
        S3_URL,
        params={"prefix": prefix, "max-keys": 1000},
        timeout=TIMEOUT
    )
    r.raise_for_status()

    root = ET.fromstring(r.content)

    keys = [
        x.text
        for x in root.findall("s3:Contents/s3:Key", NS)
        if x.text
    ]

    pattern = re.compile(
        rf"^{re.escape(ROOT)}{re.escape(symbol)}/5m/"
        rf"{re.escape(symbol)}-5m-(2026-\d{{2}})\.zip$"
    )

    return sorted({
        match.group(1)
        for key in keys
        if (match := pattern.match(key)) and match.group(1) in MONTHS
    })

def load_month(symbol, month):
    url = f"{DATA_URL}/{ROOT}{symbol}/5m/{symbol}-5m-{month}.zip"

    r = get_session().get(url, timeout=TIMEOUT)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = next(name for name in z.namelist() if name.endswith(".csv"))

        data = pd.read_csv(
            z.open(name),
            header=None,
            usecols=[0, 5],
            names=["timestamp", "volume"],
            dtype={"timestamp": "int64", "volume": "float32"}
        )

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        unit="us",
        utc=True
    )

    return data.set_index("timestamp")["volume"]


def load_symbol(symbol):
    months = get_months(symbol)

    if not months:
        return None

    data = pd.concat(
        [load_month(symbol, month) for month in months],
        copy=False
    )

    data = data[
        (~data.index.duplicated(keep="last")) &
        (data.index >= START) &
        (data.index < END)
    ].sort_index()

    if data.empty:
        return None

    return data.resample("10min").sum(min_count=2).astype("float32").rename(symbol)

symbols = get_symbols()

print(f"Símbolos históricos encontrados: {len(symbols)}")

series = {}
failed = {}

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(load_symbol, symbol): symbol
        for symbol in symbols
    }

    for i, future in enumerate(as_completed(futures), 1):
        symbol = futures[future]

        try:
            result = future.result()

            if result is not None:
                series[symbol] = result

        except Exception as e:
            failed[symbol] = str(e)

        if i % 100 == 0 or i == len(futures):
            print(
                f"{i}/{len(futures)} | "
                f"válidos: {len(series)} | "
                f"errores: {len(failed)}"
            )

if not series:
    raise RuntimeError("No se encontraron datos Spot para H1 2026")

index = pd.date_range(
    START,
    END,
    freq="10min",
    inclusive="left"
)

dfv = pd.DataFrame(series, index=index, dtype="float32")
dfv = dfv.reindex(sorted(dfv.columns), axis=1)
dfv.index.name = "timestamp"

print(f"Dimensión final: {dfv.shape}")
print(f"Pares reales: {dfv.shape[1]}")
print(f"RAM: {dfv.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Errores: {len(failed)}")

Símbolos históricos encontrados: 3695
100/3695 | válidos: 55 | errores: 0
200/3695 | válidos: 94 | errores: 0
300/3695 | válidos: 144 | errores: 0


400/3695 | válidos: 205 | errores: 0
500/3695 | válidos: 248 | errores: 0
600/3695 | válidos: 289 | errores: 0


700/3695 | válidos: 331 | errores: 0


800/3695 | válidos: 362 | errores: 0
900/3695 | válidos: 403 | errores: 0
1000/3695 | válidos: 440 | errores: 0
1100/3695 | válidos: 497 | errores: 0


1200/3695 | válidos: 538 | errores: 0


...

2300/3695 | válidos: 1052 | errores: 0
2400/3695 | válidos: 1119 | errores: 0
2500/3695 | válidos: 1167 | errores: 0
2600/3695 | válidos: 1208 | errores: 0
2700/3695 | válidos: 1253 | errores: 0
2800/3695 | válidos: 1317 | errores: 0


2900/3695 | válidos: 1375 | errores: 0


3000/3695 | válidos: 1425 | errores: 0
3100/3695 | válidos: 1474 | errores: 0


3200/3695 | válidos: 1528 | errores: 0
3300/3695 | válidos: 1576 | errores: 0
3400/3695 | válidos: 1619 | errores: 0


3500/3695 | válidos: 1671 | errores: 0
3600/3695 | válidos: 1724 | errores: 0
3695/3695 | válidos: 1788 | errores: 0
Dimensión final: (26064, 1788)
Pares reales: 1788
RAM: 177.97 MB
Errores: 0


In [ ]:
dfv[["BTCARS", "BTCTUSD", "BTCU", "BTCUAH", "BTCUSD", "BTCUSD1", "BTCUSDC", "BTCUSDS", "BTCUSDT", ]].head(5)

,BTCARS,BTCTUSD,BTCU,BTCUAH,BTCUSD,BTCUSD1,BTCUSDC,BTCUSDS,BTCUSDT
timestamp,,,,,,,,,
2026-01-01 00:00:00+00:00,0.00000,0.00000,NaN,0.0,0.00482,60.084339,22.522869,NaN,76.231766
2026-01-01 00:10:00+00:00,0.00000,0.00000,NaN,0.0,0.00191,28.948570,6.133660,NaN,19.335789
2026-01-01 00:20:00+00:00,0.00340,0.00007,NaN,0.0,0.00000,23.045200,4.083800,NaN,14.643700
2026-01-01 00:30:00+00:00,0.00875,0.00000,NaN,0.0,0.00016,26.969318,4.609730,NaN,43.847179
2026-01-01 00:40:00+00:00,0.00163,0.00000,NaN,0.0,0.00375,12.003250,4.030020,NaN,18.149521


In [ ]:
dfv[["BTCARS", "BTCBRL", "BTCDAI", "BTCEUR", "BTCEURI", "BTCFDUSD", "BTCIDR", "BTCJPY", "BTCMXN", "BTCPLN", "BTCRON", "BTCTRY", "BTCTUSD", "BTCU", "BTCUAH", "BTCUSD", "BTCUSD1", "BTCUSDC", "BTCUSDS", "BTCUSDT", "BTCZAR"]].tail(5)

,BTCARS,BTCBRL,BTCDAI,BTCEUR,BTCEURI,BTCFDUSD,BTCIDR,BTCJPY,BTCMXN,BTCPLN,...,BTCTRY,BTCTUSD,BTCU,BTCUAH,BTCUSD,BTCUSD1,BTCUSDC,BTCUSDS,BTCUSDT,BTCZAR
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-06-30 23:10:00+00:00,0.02313,0.22847,NaN,0.36697,0.01162,0.57361,0.00018,1.526266,0.036399,0.0,...,0.22103,NaN,4.29099,NaN,0.00988,4.93476,12.903720,0.00384,32.267590,NaN
2026-06-30 23:20:00+00:00,0.00690,0.37061,NaN,0.68825,0.00139,1.81589,0.00215,2.143010,0.010220,0.0,...,3.47459,NaN,2.65573,NaN,0.00000,6.36314,22.168980,0.00370,182.828293,NaN
2026-06-30 23:30:00+00:00,0.00084,0.29013,NaN,0.24604,0.00145,1.93800,0.00020,1.882686,0.010410,0.0,...,1.95648,NaN,1.37969,NaN,0.00000,5.54094,10.410410,0.00359,29.882380,NaN
2026-06-30 23:40:00+00:00,0.00412,1.29565,NaN,1.09661,0.00145,0.24379,0.09000,2.153635,0.000858,0.0,...,0.24377,NaN,3.47573,NaN,0.00329,7.21753,18.515091,0.00366,50.722271,NaN
2026-06-30 23:50:00+00:00,0.00108,0.39871,NaN,0.75864,0.00140,0.49425,0.01147,2.428935,0.000207,0.0,...,0.31086,NaN,5.43050,NaN,0.00298,6.18680,21.323851,0.00374,214.475494,NaN


In [ ]:
dfv[["BTCARS", "BTCEUR", "BTCEURI", "BTCFDUSD", "BTCIDR", "BTCJPY", "BTCMXN", "BTCPLN", "BTCRON",  "BTCUSD", "BTCUSD1", "BTCUSDC", "BTCUSDS", "BTCUSDT", "BTCZAR"]].head(10)

,BTCARS,BTCEUR,BTCEURI,BTCFDUSD,BTCIDR,BTCJPY,BTCMXN,BTCPLN,BTCRON,BTCUSD,BTCUSD1,BTCUSDC,BTCUSDS,BTCUSDT,BTCZAR
timestamp,,,,,,,,,,,,,,,
2026-01-01 00:00:00+00:00,0.00000,3.97473,0.02804,91.913300,0.09158,1.076977,0.006357,0.00927,0.00000,0.00482,60.084339,22.522869,NaN,76.231766,0.00009
2026-01-01 00:10:00+00:00,0.00000,0.39581,0.00020,42.700680,0.00206,0.322878,0.005615,0.01096,0.00000,0.00191,28.948570,6.133660,NaN,19.335789,0.00108
2026-01-01 00:20:00+00:00,0.00340,0.46217,0.00009,27.566792,0.00376,0.546515,0.008403,0.00000,0.00000,0.00000,23.045200,4.083800,NaN,14.643700,0.00053
2026-01-01 00:30:00+00:00,0.00875,0.80833,0.00030,38.968208,0.00168,1.031010,0.006154,0.00000,0.00000,0.00016,26.969318,4.609730,NaN,43.847179,0.00275
2026-01-01 00:40:00+00:00,0.00163,0.55371,0.00670,32.094059,0.00203,0.862843,0.014508,0.00000,0.00000,0.00375,12.003250,4.030020,NaN,18.149521,0.00010
2026-01-01 00:50:00+00:00,0.00752,0.43546,0.00060,57.150879,0.00172,1.417866,0.009194,0.00000,0.00000,0.00007,21.928440,6.389250,NaN,61.452400,0.00000
2026-01-01 01:00:00+00:00,0.00331,0.72580,0.00486,132.705780,0.00404,2.449695,0.004380,0.00000,0.00000,0.00000,25.179070,22.181030,NaN,58.739479,0.00000
2026-01-01 01:10:00+00:00,0.00187,0.58874,0.02732,114.325172,0.00308,2.061931,0.012798,0.00000,0.00007,0.00007,25.257700,16.792759,NaN,78.345047,0.00000
2026-01-01 01:20:00+00:00,0.00005,0.50574,0.00100,96.695938,0.00543,0.881304,0.013430,0.00827,0.00000,0.00005,23.392590,7.627470,NaN,26.937569,0.00017


In [ ]:
dfv_sorted_eur = dfv.sort_values(by='BTCEUR', ascending=False)
display(dfv_sorted_eur[['BTCEUR', 'BTCUSDT', 'BTCUSDC', 'BTCFDUSD', 'BTCJPY']].head(5))

,BTCEUR,BTCUSDT,BTCUSDC,BTCFDUSD,BTCJPY
timestamp,,,,,
2026-06-25 13:50:00+00:00,106.824249,3891.006836,948.449829,131.914795,59.254036
2026-01-31 18:40:00+00:00,96.761490,4100.879883,1051.202881,539.395264,62.637138
2026-02-06 00:10:00+00:00,86.033005,5356.250000,1275.589111,347.849670,70.628693
2026-02-06 00:00:00+00:00,83.742310,3391.296875,701.243408,166.383835,136.165421
2026-01-30 01:40:00+00:00,83.055130,3081.528809,661.266296,279.716492,52.539574


In [ ]:
dfv_sorted_jpy = dfv.sort_values(by='BTCJPY', ascending=False)
display(dfv_sorted_jpy[['BTCJPY', 'BTCUSDT', 'BTCUSDC', 'BTCFDUSD', 'BTCEUR']].head(5))

,BTCJPY,BTCUSDT,BTCUSDC,BTCFDUSD,BTCEUR
timestamp,,,,,
2026-02-06 00:00:00+00:00,136.165421,3391.296875,701.243408,166.383835,83.742310
2026-06-05 16:10:00+00:00,89.444275,2576.326172,1111.241821,125.824707,73.614212
2026-06-05 18:40:00+00:00,86.764404,725.500488,217.836212,25.930969,16.593580
2026-02-06 21:00:00+00:00,86.502945,294.789398,144.826233,47.653076,15.656560
2026-02-02 14:30:00+00:00,85.743179,726.990784,211.780273,150.175430,21.592869


In [ ]:
dfv_sorted_ars = dfv.sort_values(by='BTCARS', ascending=False)
display(dfv_sorted_ars[['BTCARS', 'BTCUSDT', 'BTCUSDC', 'BTCFDUSD', 'BTCEUR']].head(5))

,BTCARS,BTCUSDT,BTCUSDC,BTCFDUSD,BTCEUR
timestamp,,,,,
2026-05-27 21:40:00+00:00,0.44618,298.340973,92.446617,10.712990,19.338110
2026-01-28 02:20:00+00:00,0.31859,51.073372,6.165240,48.643318,0.247920
2026-01-30 01:30:00+00:00,0.30705,1468.740967,221.451202,293.977112,34.162369
2026-06-30 13:30:00+00:00,0.22377,858.149048,156.331024,24.930161,12.129190
2026-06-05 14:20:00+00:00,0.22332,710.991455,292.526794,17.912979,17.708790


In [ ]:
dfv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 1788 entries, 0GBNB to 币安人生USDT
dtypes: float32(1788)
memory usage: 177.8 MB


In [ ]:
columnas = pd.DataFrame({"columna": dfv.columns})
columnas

,columna
0,0GBNB
1,0GFDUSD
2,0GTRY
3,0GUSDC
4,0GUSDT
...,...
1783,币安人生TRY
1784,币安人生U
1785,币安人生USD1
1786,币安人生USDC


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(dfv=columnas)

https://docs.google.com/spreadsheets/d/1ukGVKUQqaI76MnsxXyJsGrVxkefcpshyidXoYNpviUc/edit#gid=0


In [ ]:
nan_por_columna = dfv.isna().sum().rename("NaNs").to_frame()
nan_por_columna = nan_por_columna[nan_por_columna["NaNs"] > 0]
nan_por_columna

,NaNs
0GBNB,23280
0GFDUSD,21840
1000SATSFDUSD,24894
1INCHBTC,9774
1MBABYDOGEFDUSD,23280
...,...
币安人生TRY,14754
币安人生U,15888
币安人生USD1,15888
币安人生USDC,948


In [ ]:
dfr = dfv[["BTCUSDT", "SOLUSDT", "BNBUSDT", "TRUUSDT", "ETHUSDT"]]
dfv.head(5)

,BTCUSDT,SOLUSDT,BNBUSDT,TRUUSDT,ETHUSDT
timestamp,,,,,
2026-01-01 00:00:00+00:00,76.231766,11724.738281,952.643066,746043.0,2590.970215
2026-01-01 00:10:00+00:00,19.335789,7117.146973,267.971008,28748.0,338.273926
2026-01-01 00:20:00+00:00,14.643700,10099.746094,322.561005,0.0,320.085693
2026-01-01 00:30:00+00:00,43.847179,8835.511719,319.096985,15680.0,511.476624
2026-01-01 00:40:00+00:00,18.149521,4205.128906,498.889008,10995.0,290.592987


## Tr Volume


*PRICE*

In [ ]:
df = df.loc[:, df.columns.str.contains("USD", case=False)]
df = df.loc[:, df.columns.str[:3].str.upper() != "USD"]
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26064 entries, 2026-01-01 00:00:00+00:00 to 2026-06-30 23:50:00+00:00
Freq: 10min
Columns: 945 entries, 0GFDUSD to 币安人生USDT
dtypes: float32(945)
memory usage: 94.2 MB


In [ ]:
df[[ "BTCFDUSD", "BTCTUSD", "BTCUSD", "BTCUSD1", "BTCUSDC", "BTCUSDS", "BTCUSDT"]].head(10)

,BTCFDUSD,BTCTUSD,BTCUSD,BTCUSD1,BTCUSDC,BTCUSDS,BTCUSDT
timestamp,,,,,,,
2026-01-01 00:00:00+00:00,87815.992188,87829.507812,87658.960938,87700.171875,87649.992188,NaN,87764.062500
2026-01-01 00:10:00+00:00,87824.718750,87829.507812,87691.007812,87705.617188,87649.148438,NaN,87760.187500
2026-01-01 00:20:00+00:00,87825.656250,87976.203125,87691.007812,87697.671875,87659.406250,NaN,87776.179688
2026-01-01 00:30:00+00:00,87797.890625,87976.203125,87641.757812,87677.656250,87637.679688,NaN,87748.101562
2026-01-01 00:40:00+00:00,87869.210938,87976.203125,87674.757812,87741.031250,87700.781250,NaN,87813.171875
2026-01-01 00:50:00+00:00,87858.187500,87976.203125,87720.171875,87732.000000,87694.023438,NaN,87809.226562
2026-01-01 01:00:00+00:00,88013.117188,87976.203125,87720.171875,87884.468750,87846.140625,NaN,87957.140625
2026-01-01 01:10:00+00:00,88001.992188,88116.953125,87846.929688,87867.312500,87832.523438,NaN,87952.851562
2026-01-01 01:20:00+00:00,87931.859375,88099.687500,87875.523438,87799.867188,87757.992188,NaN,87875.218750


In [ ]:
df = df.loc[:, df.columns.str.upper().str.endswith("USDT")]
df.info()

*VOLUME*

In [ ]:
dfv = dfv.loc[:, dfv.columns.str.contains("USD", case=False)]
dfv = dfv.loc[:, dfv.columns.str[:3].str.upper() != "USD"]
dfv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 945 entries, 0GFDUSD to 币安人生USDT
dtypes: float32(945)
memory usage: 94.0 MB


In [ ]:
dfv[[ "BTCFDUSD", "BTCTUSD", "BTCUSD", "BTCUSD1", "BTCUSDC", "BTCUSDS", "BTCUSDT"]].head(10)

,BTCFDUSD,BTCTUSD,BTCUSD,BTCUSD1,BTCUSDC,BTCUSDS,BTCUSDT
0,91.913300,0.00000,0.00482,60.084339,22.522869,NaN,76.231766
1,42.700680,0.00000,0.00191,28.948570,6.133660,NaN,19.335789
2,27.566792,0.00007,0.00000,23.045200,4.083800,NaN,14.643700
3,38.968208,0.00000,0.00016,26.969318,4.609730,NaN,43.847179
4,32.094059,0.00000,0.00375,12.003250,4.030020,NaN,18.149521
5,57.150879,0.00000,0.00007,21.928440,6.389250,NaN,61.452400
6,132.705780,0.00000,0.00000,25.179070,22.181030,NaN,58.739479
7,114.325172,0.02076,0.00007,25.257700,16.792759,NaN,78.345047
8,96.695938,0.00007,0.00005,23.392590,7.627470,NaN,26.937569
9,67.515762,0.00164,0.00033,22.142250,6.521000,NaN,24.815889


In [ ]:
# dfv = dfv.loc[:, dfv.columns.str.upper().str.endswith("USDT")]
# dfv.info()

In [ ]:
prefijos = ["0G", "1000CAT", "1000CHEEMS", "1000SATS", "1INCH", "1MBABYDOGE", "2Z",
            "A2Z", "AAVE", "ACA", "ACE", "ACH", "ACM", "ACT", "ACX", "ADA", "ADX",
            "AEUR", "AEVO", "AGLD", "AIGENSYN", "AI", "AIXBT", "ALCX", "ALGO", "ALICE",
            "ALLO", "ALPINE", "ALT", "AMDB", "AMP", "ANIME", "ANKR", "APE", "API3",
            "APT", "ARB", "ARDR", "ARKM", "ARK", "ARPA", "AR", "ASR", "ASTER",
            "ASTR", "ATA", "ATM", "ATOM", "AT", "AUCTION", "AUDIO", "A", "AVA", "AVAX", "AVNT", "AWE",
            "AXL", "AXS", "BABY", "BANANAS31", "BANANA", "BAND", "BANK", "BARD", "BAR", "BAT", "BB",
            "BCH", "BEAMX", "BEL", "BERA", "BFUSD", "BICO", "BIFI", "BIGTIME", "BIO", "BLUR", "BMT",
            "BNB", "BNSOL", "BNT", "BOME", "BONK", "BREV", "BROCCOLI714", "BTC", "BTTC", "C98", "CAKE",
            "CATI", "CELO", "CELR", "CETUS", "CFG", "CFX", "CGPT", "CHESS", "CHIP", "CHR", "CHZ", "CITY",
            "CKB", "COMP", "COOKIE", "COS", "COTI", "COW", "CRCLB", "CRV", "CTK", "CTSI", "C", "CVC", "CVX",
            "CYBER", "DASH", "DATA", "DCR", "DEGO", "DENT", "DEXE", "DF", "DGB", "DIA", "DODO", "DOGE", "DOGS",
            "DOLO", "DOT", "D", "DUSK", "DYDX", "DYM", "EDEN", "EDU", "EGLD", "EIGEN", "ENA", "ENJ", "ENSO",
            "ENS", "EPIC", "ERA", "ESP", "ETC", "ETHFI", "ETH", "EUL", "EURI", "EUR", "EWYB", "FARM", "FDUSD",
            "FET", "FF", "FIDA", "FIL", "FIO", "FLOKI", "FLOW", "FLUX", "FOGO", "FORM", "FORTH", "FRAX", "FTT",
            "FUN", "F", "FXS", "GALA", "GAS", "GENIUS", "GHST", "GIGGLE", "GLMR", "GLM", "GMT", "GMX", "GNO",
            "GNS", "GPS", "GRT", "GTC", "GUN", "G", "HAEDAL", "HBAR", "HEI", "HEMI", "HFT", "HIGH", "HIVE",
            "HMSTR", "HOLO", "HOME", "HOOK", "HOT", "HUMA", "HYPER", "ICP", "ICX", "IDEX", "ID", "ILV",
            "IMX", "INIT", "INJ", "INTCB", "IOST", "IOTA", "IOTX", "IO", "IQ", "JASMY", "JOE", "JST", "JTO",
            "JUP", "JUV", "KAIA", "KAITO", "KAT", "KAVA", "KERNEL", "KGST", "KITE", "KMNO", "KNC", "KSM", "LA",
            "LAYER", "LAZIO", "LDO", "LINEA", "LINK", "LISTA", "LITEB", "LPT", "LQTY", "LRC", "LSK", "LTC", "LUMIA",
            "LUNA", "LUNC", "MAGIC", "MANA", "MANTA", "MANTRA", "MASK", "MAV", "MBL", "MBOX", "MDT", "MEGA", "MEME",
            "METAB", "METIS", "MET", "ME", "MINA", "MIRA", "MITO", "MLN", "MMT", "MORPHO", "MOVE", "MOVR", "MSFTB",
            "MSTRB", "MTL", "MUBARAK", "MUB", "NEAR", "NEIRO", "NEO", "NEWT", "NEXO", "NFP", "NIGHT", "NIL", "NKN",
            "NMR", "NOM", "NOT", "NTRN", "NVDAB", "NXPC", "OGN", "OG", "OM", "ONDO", "ONE", "ONG", "ONT", "OPEN", "OPG",
            "OPN", "OP", "ORCA", "ORDI", "OSMO", "OXT", "PARTI", "PAXG", "PENDLE", "PENGU", "PEOPLE", "PEPE", "PHA", "PHB",
            "PIVX", "PIXEL", "PLTRB", "PLUME", "PNUT", "POL", "POLYX", "POND", "PORTAL", "PORTO", "POWR", "PROM", "PROVE",
            "PSG", "PUMP", "PUNDIX", "PYR", "PYTH", "QI", "QKC", "QNT", "QQQB", "QTUM", "QUICK", "RAD", "RARE", "RAY",
            "RDNT", "RED", "RENDER", "REQ", "RESOLV", "RE", "REZ", "RIF", "RLC", "RLUSD", "ROBO", "RONIN", "ROSE",
            "RPL", "RSR", "RUNE", "RVN", "SAGA", "SAHARA", "SAND", "SANTOS", "SAPIEN", "SCRT", "SCR", "SC", "SEI",
            "SENT", "SFP", "SHELL", "SHIB", "SIGN", "SKL", "SKY", "SLP", "SNDKB", "SNX", "SOL", "SOLV", "SOMI",
            "SOPH", "SPCXB", "SPELL", "SPK", "SSV", "STEEM", "STG", "STORJ", "STO", "STRAX", "STRK", "STX", "SUI",
            "SUN", "SUPER", "S", "SUSHI", "SXP", "SXT", "SYN", "SYRUP", "SYS", "TAO", "TFUEL", "THETA", "THE", "TIA",
            "TKO", "TLM", "TNSR", "TON", "TOWNS", "TRB", "TREE", "TRUMP", "TRU", "TRX", "TSLAB", "TST", "TURBO",
            "TURTLE", "T", "TUSD", "TUT", "TWT", "UMA", "UNI", "USTC", "USUAL", "UTK", "U", "VANA", "VANRY",
            "VELODROME", "VET", "VIC", "VIRTUAL", "VTHO", "WAL", "WAN", "WAXP", "WBETH", "WBTC", "WCT", "WIF",
            "WIN", "WLD", "WLFI", "WOO", "W", "XAI", "XAUT", "XEC", "XLM", "XNO", "XPL", "XRP", "XTZ",
            "XUSD", "XVG", "XVS", "YB", "YFI", "YGG", "ZAMA", "ZBT", "ZEC", "ZEN", "ZIL", "ZKC", "ZKP", "ZK", "ZRO", "ZRX", "币安人生"]

In [ ]:
import numpy as np
prefijos = prefijos[0] if isinstance(prefijos[0], list) else prefijos
prefijos_ord = sorted(set(prefijos), key=len, reverse=True)

asignacion = {
    col: next((p for p in prefijos_ord if col.startswith(p)), None)
    for col in dfv.columns
}

dfv = pd.DataFrame({
    f"{p}_V": dfv[[c for c, x in asignacion.items() if x == p]].sum(axis=1, min_count=1)
    if p in asignacion.values()
    else pd.Series(np.nan, index=dfv.index)
    for p in prefijos
}, index=dfv.index)

dfv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 472 entries, 0G_V to 币安人生_V
dtypes: float32(471), float64(1)
memory usage: 47.0 MB


In [ ]:
dfv[["BTC_V"]].head(10)

,BTC_V
0,250.757095
1,97.120605
2,69.339561
3,114.394592
4,66.280602
5,146.921051
6,238.805359
7,234.741501
8,154.653687
9,120.996872


***TOPOLOGIA CONTINUA USDS***

In [ ]:
df.info()
dfv.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26064 entries, 2026-01-01 00:00:00+00:00 to 2026-06-30 23:50:00+00:00
Freq: 10min
Columns: 472 entries, 0GUSDT to 币安人生USDT
dtypes: float32(472)
memory usage: 47.1 MB
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26064 entries, 2026-01-01 00:00:00+00:00 to 2026-06-30 23:50:00+00:00
Freq: 10min
Columns: 472 entries, 0G_V to 币安人生_V
dtypes: float32(471), float64(1)
memory usage: 47.2 MB


In [ ]:
prefijos_ord = sorted(set(prefijos), key=len, reverse=True)

asignacion = {
    col: next((p for p in prefijos_ord if col.startswith(p)), None)
    for col in df.columns
}

dftu = pd.DataFrame({
    f"{p}_y": df[col] * dfv[f"{p}_V"]
    for col, p in asignacion.items()
    if p is not None and f"{p}_V" in dfv.columns
}, index=df.index)

dftu.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26064 entries, 2026-01-01 00:00:00+00:00 to 2026-06-30 23:50:00+00:00
Freq: 10min
Columns: 471 entries, 0G_y to 币安人生_y
dtypes: float32(471)
memory usage: 47.0 MB


In [ ]:
dftu[["BTC_y","ETH_y", "SOL_y", "BNB_y"]].head(5)

,BTC_y,ETH_y,SOL_y,BNB_y
timestamp,,,,
2026-01-01 00:00:00+00:00,22577828.0,17213926.0,4662368.00,5386866.50
2026-01-01 00:10:00+00:00,8625638.0,4179510.0,2879007.25,2218168.50
2026-01-01 00:20:00+00:00,6241475.5,4367593.5,3085364.75,2232245.25
2026-01-01 00:30:00+00:00,10296328.0,6947561.0,2622634.00,2551085.75
2026-01-01 00:40:00+00:00,6055092.0,4563440.0,1628995.75,1810666.25


In [ ]:
dftu["YS"] = dftu.sum(axis=1, min_count=1)
dftu[["BTC_y","ETH_y", "SOL_y", "BNB_y", "YS"]].head(5)

,BTC_y,ETH_y,SOL_y,BNB_y,YS
timestamp,,,,,
2026-01-01 00:00:00+00:00,22577828.0,17213926.0,4662368.00,5386866.50,77447696.0
2026-01-01 00:10:00+00:00,8625638.0,4179510.0,2879007.25,2218168.50,31685750.0
2026-01-01 00:20:00+00:00,6241475.5,4367593.5,3085364.75,2232245.25,29428756.0
2026-01-01 00:30:00+00:00,10296328.0,6947561.0,2622634.00,2551085.75,35800816.0
2026-01-01 00:40:00+00:00,6055092.0,4563440.0,1628995.75,1810666.25,27099364.0


In [ ]:
dfyp = dftu.div(dftu["YS"], axis=0)
dfyp[["BTC_y","ETH_y", "SOL_y", "BNB_y", "YS"]].head(5)

,BTC_y,ETH_y,SOL_y,BNB_y,YS
timestamp,,,,,
2026-01-01 00:00:00+00:00,0.291524,0.222265,0.060200,0.069555,1.0
2026-01-01 00:10:00+00:00,0.272225,0.131905,0.090861,0.070005,1.0
2026-01-01 00:20:00+00:00,0.212088,0.148412,0.104842,0.075853,1.0
2026-01-01 00:30:00+00:00,0.287600,0.194062,0.073256,0.071258,1.0
2026-01-01 00:40:00+00:00,0.223440,0.168397,0.060112,0.066816,1.0


# CVS

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Mounted at /content/drive


*SAVE*

**PRECIO**

In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde querés guardar el archivo (podés cambiar el nombre o carpeta)
ruta_guardado = '/content/drive/MyDrive/0626p.parquet'

# Guardar en formato Parquet, conservando la estructura y tipos
df.to_parquet(ruta_guardado, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'df' is not defined

**VOLUME**

In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde querés guardar el archivo (podés cambiar el nombre o carpeta)
ruta_guardado = '/content/drive/MyDrive/0626v.parquet'

# Guardar en formato Parquet, conservando la estructura y tipos
dfv.to_parquet(ruta_guardado, index=False)

In [ ]:
dfv.info()

NameError: name 'dfv' is not defined

**VOLUMEN TRANSACCIONAL**



In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde querés guardar el archivo (podés cambiar el nombre o carpeta)
ruta_guardado = '/content/drive/MyDrive/0626dftu.parquet'

# Guardar en formato Parquet, conservando la estructura y tipos
dftu.to_parquet(ruta_guardado, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde querés guardar el archivo (podés cambiar el nombre o carpeta)
ruta_guardado = '/content/drive/MyDrive/0626dfyp.parquet'

# Guardar en formato Parquet, conservando la estructura y tipos
dfyp.to_parquet(ruta_guardado, index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


***APERTURA***

**TOPOLOGICAS**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dftu = pd.read_parquet('/content/drive/MyDrive/0626dftu.parquet')
dftu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 472 entries, 0G_y to YS
dtypes: float32(472)
memory usage: 46.9 MB


In [ ]:
dftu[["BTC_y","ETH_y", "SOL_y", "BNB_y", "YS"]].head(5)

,BTC_y,ETH_y,SOL_y,BNB_y,YS
0,22577828.0,17213926.0,4662368.00,5386866.50,77447696.0
1,8625638.0,4179510.0,2879007.25,2218168.50,31685750.0
2,6241475.5,4367593.5,3085364.75,2232245.25,29428756.0
3,10296328.0,6947561.0,2622634.00,2551085.75,35800816.0
4,6055092.0,4563440.0,1628995.75,1810666.25,27099364.0


In [ ]:
dfyp = pd.read_parquet('/content/drive/MyDrive/0626dfyp.parquet')
dfyp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 472 entries, 0G_y to YS
dtypes: float32(472)
memory usage: 46.9 MB


In [ ]:
dfyp = dfyp.rename(columns=lambda c: c[:-2] + "_p" if c.endswith("_y") else "YS_p" if c == "YS" else c)

In [ ]:
dfyp[["BTC_p","ETH_p", "SOL_p", "BNB_p", "YS_p"]].head(5)

,BTC_p,ETH_p,SOL_p,BNB_p,YS_p
0,0.291524,0.222265,0.060200,0.069555,1.0
1,0.272225,0.131905,0.090861,0.070005,1.0
2,0.212088,0.148412,0.104842,0.075853,1.0
3,0.287600,0.194062,0.073256,0.071258,1.0
4,0.223440,0.168397,0.060112,0.066816,1.0


**PRICE**

In [ ]:
dfp = pd.read_parquet('/content/drive/MyDrive/0626p.parquet')
dfp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 1788 entries, 0GBNB to 币安人生USDT
dtypes: float32(1788)
memory usage: 177.8 MB


In [ ]:
dfp = dfp.loc[:, dfp.columns.str.contains("USD", case=False)]
dfp = dfp.loc[:, dfp.columns.str[:3].str.upper() != "USD"]
dfp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 945 entries, 0GFDUSD to 币安人生USDT
dtypes: float32(945)
memory usage: 94.0 MB


In [ ]:
dfp = dfp.loc[:, dfp.columns.str.upper().str.endswith("USDT")]
dfp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 472 entries, 0GUSDT to 币安人生USDT
dtypes: float32(472)
memory usage: 46.9 MB


In [ ]:
dfp[["BTCUSDT","ETHUSDT", "SOLUSDT", "BNBUSDT"]].head(5)

,BTCUSDT,ETHUSDT,SOLUSDT,BNBUSDT
0,87764.062500,2978.080078,125.010002,865.099976
1,87760.187500,2977.620117,124.949997,865.719971
2,87776.179688,2979.750000,125.160004,866.289978
3,87748.101562,2979.560059,125.110001,866.049988
4,87813.171875,2981.840088,125.230003,866.070007


**VOLUME**

In [ ]:
dfv = pd.read_parquet('/content/drive/MyDrive/0626v.parquet')
dfv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Columns: 472 entries, 0G_V to 币安人生_V
dtypes: float32(471), float64(1)
memory usage: 47.0 MB


In [ ]:
dfv.columns = dfv.columns.str.replace("_V$", "_v", regex=True)
dfv.head()

,0G_v,1000CAT_v,1000CHEEMS_v,1000SATS_v,1INCH_v,1MBABYDOGE_v,2Z_v,A2Z_v,AAVE_v,ACA_v,...,ZBT_v,ZEC_v,ZEN_v,ZIL_v,ZKC_v,ZKP_v,ZK_v,ZRO_v,ZRX_v,币安人生_v
0,27694.121094,1.472378e+07,12122814.0,347160160.0,9574.600586,10059754.0,208117.0,2006481.0,827.527954,1.082632e+06,...,852224.93750,861.927979,7224.319824,715211.500000,33755.898438,NaN,820042.250000,6710.060059,251131.0,NaN
1,9044.120117,7.249850e+06,8148085.0,211520160.0,68690.898438,7738302.0,523764.0,1487363.0,145.222000,7.663925e+04,...,518803.71875,441.914978,2241.429932,44003.597656,21907.898438,NaN,608021.500000,11220.870117,49380.0,NaN
2,12262.150391,1.920092e+06,5410546.0,58933792.0,11286.700195,36664248.0,432896.0,789411.0,163.569992,1.190682e+06,...,577796.00000,738.122009,3108.980225,426629.906250,10816.399414,NaN,993517.875000,3602.959961,139067.0,NaN
3,21014.769531,3.440486e+05,5147898.0,314197600.0,112238.000000,7317814.0,152057.0,1404740.0,159.023987,5.765492e+04,...,285229.50000,569.263000,2742.040039,331618.906250,210629.500000,NaN,333615.312500,1677.709961,178188.0,NaN
4,22442.148438,2.956485e+06,6005280.0,669291072.0,24635.099609,1906056.0,232514.0,702542.0,154.630997,1.124021e+06,...,360595.12500,177.720993,1520.849976,290649.687500,18457.201172,NaN,169226.890625,3992.780029,236083.0,NaN


In [ ]:
dfv[["BTC_v"]].head(5)

,BTC_v
0,250.757095
1,97.120605
2,69.339561
3,114.394592
4,66.280602


In [ ]:
# 10 mins
# BTCUSDT: PRECIO
# BTC_y: SIMPLEX PROBABILISTICO

---

# Y

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.concat([dfyp, dfp, dftu, dfv], axis=1)
df[["BTCUSDT", "BTC_y", "BTC_p", "BTC_v"]].head(5)

,BTCUSDT,BTC_y,BTC_p,BTC_v
0,87764.062500,22577828.0,0.291524,250.757095
1,87760.187500,8625638.0,0.272225,97.120605
2,87776.179688,6241475.5,0.212088,69.339561
3,87748.101562,10296328.0,0.287600,114.394592
4,87813.171875,6055092.0,0.223440,66.280602


Y

In [ ]:
Y = df[['BTC_y', "BTCUSDT"]].shift(-1)
Y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26064 entries, 0 to 26063
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   BTC_y    26063 non-null  float32
 1   BTCUSDT  26063 non-null  float32
dtypes: float32(2)
memory usage: 203.8 KB


In [ ]:
df['BTC_yo'] = df['BTC_p'].shift(-1)

In [ ]:
df['BTCUSDT_yo'] = df['BTCUSDT'].shift(-1)

#X

In [ ]:
df_train = df.iloc[:-2016]
df_test = df.iloc[-2016:]

In [ ]:
print(df_train.shape, df_test.shape)

(24048, 1890) (2016, 1890)


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24048 entries, 0 to 24047
Columns: 1890 entries, 0G_p to BTCUSDT_yo
dtypes: float32(1889), float64(1)
memory usage: 173.5 MB


##simplex

### Shannon

In [ ]:
df_train["BTC_shannon"] = np.where(df_train["BTC_p"].eq(0),
                                   0,
                                   -np.log(df_train["BTC_p"])
                                   )

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [ "BTC_shannon"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

      Variable  Mutual Information
0  BTC_shannon             0.19064


In [ ]:



# from sklearn.feature_selection import mutual_info_regression
# import pandas as pd

# variables = [ "BTC_shannon"]

# data = df_train[variables + ["BTC_p"]].dropna()

# X = data[variables]
# y = data["BTC_p"]

# mi = mutual_info_regression(X, y, random_state=42)

# mi_df = pd.DataFrame({
#     "Variable": variables,
#     "Mutual Information": mi
# }).sort_values("Mutual Information", ascending=False)

# print(mi_df)

###EMAS

In [ ]:
ema_spans = {
    "20": 2,
    "30": 3,
    "1h": 6,
    "2h": 12,
    "4h": 24,
    "6h": 36,
    "12h": 72,
    "1d": 144,
    "3d": 432,
    "1s": 1008,
    "2s": 2016,
    "1m": 4032,
    "2m": 8064,
    "3m": 12128,
}

# Calcular cada EMA
for label, span in ema_spans.items():
    df_train[f"EMA_{label}"] = df_train["BTCUSDT"].ewm(span=span, adjust=False).mean()

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["EMA_20", "EMA_30", "EMA_1h", "EMA_2h", "EMA_4h", "EMA_6h",
             "EMA_12h", "EMA_1d", "EMA_3d", "EMA_1s", "EMA_2s", "EMA_1m", "EMA_2m", "EMA_3m"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

   Variable  Mutual Information
11   EMA_1m            0.121643
13   EMA_3m            0.117829
9    EMA_1s            0.114168
7    EMA_1d            0.108216
10   EMA_2s            0.103877
8    EMA_3d            0.102706
12   EMA_2m            0.097398
2    EMA_1h            0.096984
5    EMA_6h            0.094486
6   EMA_12h            0.094361
4    EMA_4h            0.091913
3    EMA_2h            0.091432
1    EMA_30            0.088910
0    EMA_20            0.086461


###neguentropy index

In [ ]:
df_train["nBTC"] = df_train["EMA_1m"]/df_train["BTC_shannon"]

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["nBTC"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

  Variable  Mutual Information
0     nBTC            0.171078


In [ ]:
df_train["nBTC"] = df_train["EMA_1m"]/df_train["BTC_shannon"]

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["nBTC"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

  Variable  Mutual Information
0     nBTC            0.171078


###MACD 20

In [ ]:
# MACD 20
bases = ["20"]

# Generar MACD y Señal para cada base comparada con EMAs de plazos mayores
for base in bases:
    base_span = ema_spans[base]
    for label, span in ema_spans.items():
        if span > base_span:
            macd_col = f"MACD_{base}_{label}"
            signal_col = f"Señal_{base}_{label}"
            df_train[macd_col] = df_train[f"EMA_{base}"] - df_train[f"EMA_{label}"]
            df_train[signal_col] = df_train[macd_col].ewm(span=9, adjust=False).mean()

In [ ]:
print(df_train.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["MACD_20_30", "MACD_20_1h", "MACD_20_2h", "MACD_20_4h", "MACD_20_6h", "MACD_20_12h",
             "MACD_20_1d", "MACD_20_3d", "MACD_20_1s", "MACD_20_2s", "MACD_20_1m"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

       Variable  Mutual Information
10   MACD_20_1m            0.060069
4    MACD_20_6h            0.060021
2    MACD_20_2h            0.057975
9    MACD_20_2s            0.057619
1    MACD_20_1h            0.057256
0    MACD_20_30            0.056608
5   MACD_20_12h            0.051666
8    MACD_20_1s            0.045423
3    MACD_20_4h            0.045209
7    MACD_20_3d            0.039603
6    MACD_20_1d            0.032452


###MACD 30

In [ ]:
# MACD 30
bases = ["30"]

# Generar MACD y Señal para cada base comparada con EMAs de plazos mayores
for base in bases:
    base_span = ema_spans[base]
    for label, span in ema_spans.items():
        if span > base_span:
            macd_col = f"MACD_{base}_{label}"
            signal_col = f"Señal_{base}_{label}"
            df_train[macd_col] = df_train[f"EMA_{base}"] - df_train[f"EMA_{label}"]
            df_train[signal_col] = df_train[macd_col].ewm(span=9, adjust=False).mean()

In [ ]:
print(df_train.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["MACD_30_1h", "MACD_30_2h", "MACD_30_4h", "MACD_30_6h", "MACD_30_12h",
             "MACD_30_1d", "MACD_30_3d", "MACD_30_1s", "MACD_30_2s", "MACD_30_1m"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

      Variable  Mutual Information
1   MACD_30_2h            0.061342
9   MACD_30_1m            0.060971
3   MACD_30_6h            0.057300
2   MACD_30_4h            0.056381
0   MACD_30_1h            0.056067
8   MACD_30_2s            0.052393
4  MACD_30_12h            0.042987
5   MACD_30_1d            0.042237
7   MACD_30_1s            0.039654
6   MACD_30_3d            0.038800


###MACD 1h

In [ ]:
# MACD 1h
bases = ["1h"]

# Generar MACD y Señal para cada base comparada con EMAs de plazos mayores
for base in bases:
    base_span = ema_spans[base]
    for label, span in ema_spans.items():
        if span > base_span:
            macd_col = f"MACD_{base}_{label}"
            signal_col = f"Señal_{base}_{label}"
            df_train[macd_col] = df_train[f"EMA_{base}"] - df_train[f"EMA_{label}"]
            df_train[signal_col] = df_train[macd_col].ewm(span=9, adjust=False).mean()

In [ ]:
print(df_train.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [ "MACD_1h_2h", "MACD_1h_4h", "MACD_1h_6h", "MACD_1h_12h",
             "MACD_1h_1d", "MACD_1h_3d", "MACD_1h_1s", "MACD_1h_2s", "MACD_1h_1m"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

      Variable  Mutual Information
8   MACD_1h_1m            0.068146
0   MACD_1h_2h            0.061394
7   MACD_1h_2s            0.053677
1   MACD_1h_4h            0.053498
2   MACD_1h_6h            0.051757
4   MACD_1h_1d            0.047940
3  MACD_1h_12h            0.046814
6   MACD_1h_1s            0.039689
5   MACD_1h_3d            0.031624


###MACD 1s

In [ ]:
# MACD 1h
bases = ["1s"]

# Generar MACD y Señal para cada base comparada con EMAs de plazos mayores
for base in bases:
    base_span = ema_spans[base]
    for label, span in ema_spans.items():
        if span > base_span:
            macd_col = f"MACD_{base}_{label}"
            signal_col = f"Señal_{base}_{label}"
            df_train[macd_col] = df_train[f"EMA_{base}"] - df_train[f"EMA_{label}"]
            df_train[signal_col] = df_train[macd_col].ewm(span=9, adjust=False).mean()

In [ ]:
print(df_train.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = ["MACD_1s_2s", "MACD_1s_1m", "MACD_1s_2m", "MACD_1s_3m"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

     Variable  Mutual Information
1  MACD_1s_1m            0.102830
2  MACD_1s_2m            0.101799
0  MACD_1s_2s            0.090952
3  MACD_1s_3m            0.090149


###RSI

In [ ]:
# Cálculo de deltas
delta = df['BTCUSDT'].diff()

# Diccionario de ventanas
ventanas_vol = {
     "20": 2,
    "30": 3,
    "1h": 6,
    "2h": 12,
    "4h": 24,
    "6h": 36,
    "12h": 72,
    "1d": 144,
    "3d": 432,
    "1s": 1008,
    "2s": 2016,
    "1m": 4032,
    "2m": 8064,
    "3m": 12128,
}

# RSI con máximas variaciones
for label, n in ventanas_vol.items():
    gain = np.sqrt((delta.where(delta > 0, 0.0).rolling(window=n).max()))
    loss = np.sqrt(((-delta.where(delta < 0, 0.0)).rolling(window=n).max()))

    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    df[f'RSIp_{label}'] = rsi


In [ ]:
print(df.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Lista de variables
variables = ['RSIp_20', 'RSIp_30', 'RSIp_1h', 'RSIp_2h', 'RSIp_4h', 'RSIp_6h',
             'RSIp_12h', 'RSIp_1d', 'RSIp_3d', 'RSIp_1s', 'RSIp_2s', 'RSIp_1m', 'RSIp_2m', 'RSIp_3m']


# Preprocesamiento: reemplazo de NaN con 0 (puede ajustar si preferís otra estrategia)
X = df[variables].fillna(0)
y = df['BTC_yo'].fillna(1).astype(int)

# Cálculo de información mutua
mi = mutual_info_classif(X, y)

# Mostrar resultados como DataFrame ordenado
mi_df = pd.DataFrame({'Variable': variables, 'Mutual Information': mi})
mi_df = mi_df.sort_values(by='Mutual Information', ascending=False)

print(mi_df)

    Variable  Mutual Information
13   RSIp_3m            0.001496
12   RSIp_2m            0.000940
0    RSIp_20            0.000614
11   RSIp_1m            0.000556
10   RSIp_2s            0.000288
1    RSIp_30            0.000134
9    RSIp_1s            0.000096
7    RSIp_1d            0.000019
8    RSIp_3d            0.000019
2    RSIp_1h            0.000000
5    RSIp_6h            0.000000
4    RSIp_4h            0.000000
3    RSIp_2h            0.000000
6   RSIp_12h            0.000000


In [ ]:
# logicamente son todos malos porque no tiene sentido ver la probabilidad desde un
# esquema de ganancias.

###MOM

In [ ]:
# Diccionario de ventanas
ventanas_vol = {
     "20": 2,
    "30": 3,
    "1h": 6,
    "2h": 12,
    "4h": 24,
    "6h": 36,
    "12h": 72,
    "1d": 144,
    "3d": 432,
    "1s": 1008,
    "2s": 2016,
    "1m": 4032,
    "2m": 8064,
    "3m": 12128,
}

# Calcular MOM% para cada ventana
for label, n in ventanas_vol.items():
    close_tn = df["BTC_p"].shift(n)
    mom = (df["BTC_p"] - close_tn) / close_tn
    df[f"MOM_{label}"] = mom


In [ ]:
print(df.columns.tolist())

['0G_p', '1000CAT_p', '1000CHEEMS_p', '1000SATS_p', '1INCH_p', '1MBABYDOGE_p', '2Z_p', 'A2Z_p', 'AAVE_p', 'ACA_p', 'ACE_p', 'ACH_p', 'ACM_p', 'ACT_p', 'ACX_p', 'ADA_p', 'ADX_p', 'AEUR_p', 'AEVO_p', 'AGLD_p', 'AIGENSYN_p', 'AI_p', 'AIXBT_p', 'ALCX_p', 'ALGO_p', 'ALICE_p', 'ALLO_p', 'ALPINE_p', 'ALT_p', 'AMDB_p', 'AMP_p', 'ANIME_p', 'ANKR_p', 'APE_p', 'API3_p', 'APT_p', 'ARB_p', 'ARDR_p', 'ARKM_p', 'ARK_p', 'ARPA_p', 'AR_p', 'ASR_p', 'ASTER_p', 'ASTR_p', 'ATA_p', 'ATM_p', 'ATOM_p', 'AT_p', 'AUCTION_p', 'AUDIO_p', 'A_p', 'AVA_p', 'AVAX_p', 'AVNT_p', 'AWE_p', 'AXL_p', 'AXS_p', 'BABY_p', 'BANANAS31_p', 'BANANA_p', 'BAND_p', 'BANK_p', 'BARD_p', 'BAR_p', 'BAT_p', 'BB_p', 'BCH_p', 'BEAMX_p', 'BEL_p', 'BERA_p', 'BFUSD_p', 'BICO_p', 'BIFI_p', 'BIGTIME_p', 'BIO_p', 'BLUR_p', 'BMT_p', 'BNB_p', 'BNSOL_p', 'BNT_p', 'BOME_p', 'BONK_p', 'BREV_p', 'BROCCOLI714_p', 'BTC_p', 'BTTC_p', 'C98_p', 'CAKE_p', 'CATI_p', 'CELO_p', 'CELR_p', 'CETUS_p', 'CFG_p', 'CFX_p', 'CGPT_p', 'CHESS_p', 'CHIP_p', 'CHR_p', 'CH

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Lista de variables
variables = ['MOM_20', 'MOM_30', 'MOM_1h', 'MOM_2h', 'MOM_4h', 'MOM_6h', 'MOM_12h',
             'MOM_1d', 'MOM_3d', 'MOM_1s', 'MOM_2s', 'MOM_1m', 'MOM_2m', 'MOM_3m']


# Preprocesamiento: reemplazo de NaN con 0 (puede ajustar si preferís otra estrategia)
X = df[variables].fillna(0)
y = df['BTC_yo'].fillna(1).astype(int)

# Cálculo de información mutua
mi = mutual_info_classif(X, y)

# Mostrar resultados como DataFrame ordenado
mi_df = pd.DataFrame({'Variable': variables, 'Mutual Information': mi})
mi_df = mi_df.sort_values(by='Mutual Information', ascending=False)

print(mi_df)


   Variable  Mutual Information
0    MOM_20                   0
1    MOM_30                   0
2    MOM_1h                   0
3    MOM_2h                   0
4    MOM_4h                   0
5    MOM_6h                   0
6   MOM_12h                   0
7    MOM_1d                   0
8    MOM_3d                   0
9    MOM_1s                   0
10   MOM_2s                   0
11   MOM_1m                   0
12   MOM_2m                   0
13   MOM_3m                   0


In [ ]:
# Era plausible pesanr que el momentum tenia valor predictivo sobre la probabilidad
# de todas formas, no lo tiene, probablemente si se envoca a la dimesion topologica tenga mas sentido

### Resistencia al impacto


In [ ]:
# Resistencia al impacto: volumen transaccional absorbido por unidad de movimiento del precio
df_train = df_train.copy()
ventanas_resistencia = ventanas_vol.copy()
eps = 1e-12

btc_log_price = np.log(df_train["BTCUSDT"].where(df_train["BTCUSDT"] > 0))

for label, n in ventanas_resistencia.items():
    volumen_acumulado = df_train["BTC_y"].clip(lower=0).rolling(n, min_periods=n).sum()
    movimiento = (btc_log_price - btc_log_price.shift(n)).abs()
    df_train[f"RIMP_{label}"] = np.log1p(volumen_acumulado) / (movimiento + eps)


In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [f"RIMP_{label}" for label in ventanas_resistencia]

data = df_train[variables + ["BTC_yo"]].replace([np.inf, -np.inf], np.nan).dropna()
X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)


    Variable  Mutual Information
10   RIMP_2s            0.042717
5    RIMP_6h            0.036911
7    RIMP_1d            0.035006
9    RIMP_1s            0.030838
13   RIMP_3m            0.029493
6   RIMP_12h            0.029037
8    RIMP_3d            0.028507
11   RIMP_1m            0.022274
12   RIMP_2m            0.021524
4    RIMP_4h            0.000341
0    RIMP_20            0.000328
2    RIMP_1h            0.000273
3    RIMP_2h            0.000242
1    RIMP_30            0.000222


### Estabilidad bajista


In [ ]:
# Estabilidad bajista: inversa acotada de la semivolatilidad negativa
btc_return = np.log(df_train["BTCUSDT"].where(df_train["BTCUSDT"] > 0)).diff()
retorno_bajista = btc_return.clip(upper=0)

for label, n in ventanas_resistencia.items():
    semivolatilidad = np.sqrt(
        retorno_bajista.pow(2).rolling(n, min_periods=n).mean()
    )
    df_train[f"RSTAB_{label}"] = 1.0 / (1.0 + semivolatilidad)


In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [f"RSTAB_{label}" for label in ventanas_resistencia]

data = df_train[variables + ["BTC_yo"]].replace([np.inf, -np.inf], np.nan).dropna()
X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)


     Variable  Mutual Information
10   RSTAB_2s            0.099349
11   RSTAB_1m            0.098847
12   RSTAB_2m            0.089693
13   RSTAB_3m            0.087696
9    RSTAB_1s            0.087658
7    RSTAB_1d            0.081708
8    RSTAB_3d            0.076209
5    RSTAB_6h            0.076054
4    RSTAB_4h            0.074518
6   RSTAB_12h            0.072359
3    RSTAB_2h            0.049937
2    RSTAB_1h            0.044085
0    RSTAB_20            0.032553
1    RSTAB_30            0.031303


### Persistencia de liquidez


In [ ]:
# Persistencia de liquidez: cuota media penalizada por inestabilidad del volumen
for label, n in ventanas_resistencia.items():
    cuota_media = df_train["BTC_p"].rolling(n, min_periods=n).mean()
    volumen_medio = df_train["BTC_y"].clip(lower=0).rolling(n, min_periods=n).mean()
    volumen_std = df_train["BTC_y"].clip(lower=0).rolling(n, min_periods=n).std()
    coef_variacion = volumen_std / (volumen_medio + eps)
    df_train[f"RLIQ_{label}"] = cuota_media / (1.0 + coef_variacion)


In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [f"RLIQ_{label}" for label in ventanas_resistencia]

data = df_train[variables + ["BTC_yo"]].replace([np.inf, -np.inf], np.nan).dropna()
X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)


    Variable  Mutual Information
0    RLIQ_20            0.164773
1    RLIQ_30            0.162447
2    RLIQ_1h            0.146120
3    RLIQ_2h            0.125537
4    RLIQ_4h            0.109655
5    RLIQ_6h            0.103297
12   RLIQ_2m            0.080624
6   RLIQ_12h            0.073498
11   RLIQ_1m            0.073231
9    RLIQ_1s            0.072656
8    RLIQ_3d            0.065419
7    RLIQ_1d            0.065288
10   RLIQ_2s            0.055917
13   RLIQ_3m            0.053642


### Fortaleza relativa


In [ ]:
# Fortaleza relativa: rendimiento de BTC frente al retorno mediano del sistema
price_cols = [
    c for c in df_train.columns
    if c.endswith("USDT") and not c.endswith("_yo")
]

retornos_sistema = np.log(df_train[price_cols].where(df_train[price_cols] > 0)).diff()
retorno_mercado = retornos_sistema.median(axis=1, skipna=True)
retorno_btc = retornos_sistema["BTCUSDT"]

for label, n in ventanas_resistencia.items():
    exceso_acumulado = (retorno_btc - retorno_mercado).rolling(n, min_periods=n).sum()
    escala = retorno_mercado.rolling(n, min_periods=n).std()
    z_relativo = exceso_acumulado / (escala + eps)
    df_train[f"RSTR_{label}"] = 0.5 + 0.5 * np.tanh(z_relativo)


In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [f"RSTR_{label}" for label in ventanas_resistencia]

data = df_train[variables + ["BTC_yo"]].replace([np.inf, -np.inf], np.nan).dropna()
X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)


    Variable  Mutual Information
8    RSTR_3d            0.010803
6   RSTR_12h            0.008026
10   RSTR_2s            0.005301
7    RSTR_1d            0.004913
9    RSTR_1s            0.002616
11   RSTR_1m            0.000294
12   RSTR_2m            0.000294
13   RSTR_3m            0.000294
5    RSTR_6h            0.000124
4    RSTR_4h            0.000000
0    RSTR_20            0.000000
3    RSTR_2h            0.000000
2    RSTR_1h            0.000000
1    RSTR_30            0.000000


##Price

### Price

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import pandas as pd

variables = [ "BTCUSDT"]

data = df_train[variables + ["BTC_yo"]].dropna()

X = data[variables]
y = data["BTC_yo"]

mi = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Variable": variables,
    "Mutual Information": mi
}).sort_values("Mutual Information", ascending=False)

print(mi_df)

  Variable  Mutual Information
0  BTCUSDT            0.081792


## Preparación de modelos


In [ ]:
# Preparación común y libre de look-ahead para todos los modelos
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

HORIZON = 1       # t+1 = 10 minutos
TEST_SIZE = 2016  # mismo corte temporal utilizado previamente
AR_LAGS = 6       # una hora de historia en barras de 10 minutos
MI_THRESHOLD = 0.09
EPS = 1e-12

model_df = df.copy()
model_df["BTC_yo_model"] = model_df["BTC_p"].shift(-HORIZON)

# Variables autoregresivas: lag 0 es la cuota disponible en t
for lag in range(AR_LAGS):
    model_df[f"BTC_p_lag{lag}"] = model_df["BTC_p"].shift(lag)

# Sorpresa de Shannon finita; epsilon evita asignar sorpresa cero cuando p=0
model_df["BTC_shannon_model"] = -np.log(model_df["BTC_p"].clip(lower=EPS, upper=1.0))

# Ventanas ya expresadas en barras de 10 minutos
ventanas_modelo = ventanas_vol.copy()

# EMA y MACD
for label, span in ventanas_modelo.items():
    model_df[f"EMA_{label}_model"] = model_df["BTCUSDT"].ewm(span=span, adjust=False).mean()

model_df["nBTC_model"] = (
    model_df["EMA_1m_model"] / model_df["BTC_shannon_model"].clip(lower=EPS)
)

for base in ["20", "30", "1h", "1s"]:
    base_span = ventanas_modelo[base]
    for label, span in ventanas_modelo.items():
        if span > base_span:
            model_df[f"MACD_{base}_{label}_model"] = (
                model_df[f"EMA_{base}_model"] - model_df[f"EMA_{label}_model"]
            )

# Retornos, RSI empleado en el notebook y momentum de la cuota
btc_log_price = np.log(model_df["BTCUSDT"].where(model_df["BTCUSDT"] > 0))
btc_return = btc_log_price.diff()
delta_price = model_df["BTCUSDT"].diff()

for label, n in ventanas_modelo.items():
    gain = np.sqrt(delta_price.where(delta_price > 0, 0.0).rolling(n).max())
    loss = np.sqrt((-delta_price.where(delta_price < 0, 0.0)).rolling(n).max())
    rs = gain / loss.replace(0, np.nan)
    model_df[f"RSIp_{label}_model"] = 100 - (100 / (1 + rs))
    model_df[f"MOM_{label}_model"] = (
        model_df["BTC_p"] - model_df["BTC_p"].shift(n)
    ) / model_df["BTC_p"].shift(n).replace(0, np.nan)

# Factores de resistencia
price_cols = [
    c for c in model_df.columns
    if c.endswith("USDT") and not c.endswith("_yo")
]
retornos_sistema = np.log(model_df[price_cols].where(model_df[price_cols] > 0)).diff()
retorno_mercado = retornos_sistema.median(axis=1, skipna=True)
retorno_bajista = btc_return.clip(upper=0)

for label, n in ventanas_modelo.items():
    volumen = model_df["BTC_y"].clip(lower=0)
    volumen_acumulado = volumen.rolling(n, min_periods=n).sum()
    movimiento = (btc_log_price - btc_log_price.shift(n)).abs()
    model_df[f"RIMP_{label}_model"] = np.log1p(volumen_acumulado) / (movimiento + EPS)

    semivolatilidad = np.sqrt(retorno_bajista.pow(2).rolling(n, min_periods=n).mean())
    model_df[f"RSTAB_{label}_model"] = 1.0 / (1.0 + semivolatilidad)

    cuota_media = model_df["BTC_p"].rolling(n, min_periods=n).mean()
    volumen_medio = volumen.rolling(n, min_periods=n).mean()
    volumen_std = volumen.rolling(n, min_periods=n).std()
    coef_variacion = volumen_std / (volumen_medio + EPS)
    model_df[f"RLIQ_{label}_model"] = cuota_media / (1.0 + coef_variacion)

    exceso = (btc_return - retorno_mercado).rolling(n, min_periods=n).sum()
    escala = retorno_mercado.rolling(n, min_periods=n).std()
    model_df[f"RSTR_{label}_model"] = 0.5 + 0.5 * np.tanh(exceso / (escala + EPS))

split_pos = len(model_df) - TEST_SIZE
train_mask = np.arange(len(model_df)) < split_pos
test_mask = np.arange(len(model_df)) >= split_pos

ar_features = [f"BTC_p_lag{lag}" for lag in range(AR_LAGS)]
candidate_features = [
    c for c in model_df.columns
    if c.endswith("_model")
    and c != "BTC_yo_model"
    and c not in ar_features
]

def evaluar_modelo(nombre, y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, 1.0)
    y_true = np.asarray(y_true, dtype=float)
    return {
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "N_test": len(y_true),
    }

resultados_modelos = []
print(f"Train hasta posición {split_pos - 1}; test desde {split_pos}.")
print(f"Candidatas para selección MI: {len(candidate_features)}")


/tmp/ipykernel_4849/2319869637.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[f"RSTR_{label}_model"] = 0.5 + 0.5 * np.tanh(exceso / (escala + EPS))
/tmp/ipykernel_4849/2319869637.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[f"RIMP_{label}_model"] = np.log1p(volumen_acumulado) / (movimiento + EPS)
/tmp/ipykernel_4849/2319869637.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

Train hasta posición 24047; test desde 24048.
Candidatas para selección MI: 140


/tmp/ipykernel_4849/2319869637.py:85: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[f"RSTR_{label}_model"] = 0.5 + 0.5 * np.tanh(exceso / (escala + EPS))
/tmp/ipykernel_4849/2319869637.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_df[f"RIMP_{label}_model"] = np.log1p(volumen_acumulado) / (movimiento + EPS)
/tmp/ipykernel_4849/2319869637.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

## Persistencia ingenua


In [ ]:
# Benchmark: la cuota de t predice directamente la cuota de t+1
cols_persistencia = ["BTC_p", "BTC_yo_model"]
persist_test = model_df.loc[test_mask, cols_persistencia].dropna()

y_test_persist = persist_test["BTC_yo_model"]
pred_persist = persist_test["BTC_p"]

resultado_persistencia = evaluar_modelo(
    "Persistencia ingenua", y_test_persist, pred_persist
)
resultados_modelos.append(resultado_persistencia)
pd.DataFrame([resultado_persistencia])


,Modelo,MAE,RMSE,R2,N_test
0,Persistencia ingenua,0.067986,0.090405,-0.104054,2015


## AR


In [ ]:
# AR lineal con seis rezagos de la cuota
ar_cols = ar_features + ["BTC_yo_model"]
ar_train = model_df.loc[train_mask, ar_cols].replace([np.inf, -np.inf], np.nan).dropna()
ar_test = model_df.loc[test_mask, ar_cols].replace([np.inf, -np.inf], np.nan).dropna()

ar_model = LinearRegression()
ar_model.fit(ar_train[ar_features], ar_train["BTC_yo_model"])
pred_ar = np.clip(ar_model.predict(ar_test[ar_features]), 0.0, 1.0)

resultado_ar = evaluar_modelo("AR", ar_test["BTC_yo_model"], pred_ar)
resultados_modelos.append(resultado_ar)

coef_ar = pd.DataFrame({
    "Variable": ar_features,
    "Coeficiente": ar_model.coef_
}).sort_values("Coeficiente", key=np.abs, ascending=False)

display(pd.DataFrame([resultado_ar]))
display(coef_ar)


,Modelo,MAE,RMSE,R2,N_test
0,AR,0.058628,0.075802,0.223815,2015


,Variable,Coeficiente
0,BTC_p_lag0,0.342010
1,BTC_p_lag1,0.119328
5,BTC_p_lag5,0.093562
2,BTC_p_lag2,0.092315
3,BTC_p_lag3,0.072275
4,BTC_p_lag4,0.064351


## ARX


In [ ]:
# Selección MI > 0.09 calculada exclusivamente sobre el periodo de entrenamiento
mi_rows = []

for variable in candidate_features:
    subset = model_df.loc[train_mask, [variable, "BTC_yo_model"]]
    subset = subset.replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) < 100 or subset[variable].nunique() < 2:
        continue
    mi_value = mutual_info_regression(
        subset[[variable]], subset["BTC_yo_model"], random_state=42
    )[0]
    mi_rows.append({"Variable": variable, "Mutual Information": mi_value})

mi_selection = pd.DataFrame(mi_rows).sort_values(
    "Mutual Information", ascending=False
)
selected_mi = mi_selection.loc[
    mi_selection["Mutual Information"] > MI_THRESHOLD, "Variable"
].tolist()

if not selected_mi:
    raise RuntimeError("Ninguna variable superó el umbral de información mutua.")

print(f"Variables seleccionadas con MI > {MI_THRESHOLD}: {len(selected_mi)}")
display(mi_selection.head(50))

arx_features = ar_features + selected_mi
arx_cols = arx_features + ["BTC_yo_model"]
arx_train = model_df.loc[train_mask, arx_cols].replace([np.inf, -np.inf], np.nan).dropna()
arx_test = model_df.loc[test_mask, arx_cols].replace([np.inf, -np.inf], np.nan).dropna()

arx_model = Pipeline([
    ("scaler", StandardScaler()),
    ("linear", LinearRegression())
])
arx_model.fit(arx_train[arx_features], arx_train["BTC_yo_model"])
pred_arx = np.clip(arx_model.predict(arx_test[arx_features]), 0.0, 1.0)

resultado_arx = evaluar_modelo("ARX MI>0.09", arx_test["BTC_yo_model"], pred_arx)
resultados_modelos.append(resultado_arx)

coef_arx = pd.DataFrame({
    "Variable": arx_features,
    "Coeficiente estandarizado": arx_model.named_steps["linear"].coef_
}).sort_values("Coeficiente estandarizado", key=np.abs, ascending=False)

display(pd.DataFrame([resultado_arx]))
display(coef_arx.head(50))


Variables seleccionadas con MI > 0.09: 32


,Variable,Mutual Information
0,BTC_shannon_model,0.190640
90,RLIQ_30_model,0.189774
86,RLIQ_20_model,0.175192
94,RLIQ_1h_model,0.173159
15,nBTC_model,0.171078
98,RLIQ_2h_model,0.146600
12,EMA_1m_model,0.121643
102,RLIQ_4h_model,0.120654
14,EMA_3m_model,0.117829
10,EMA_1s_model,0.114167


,Modelo,MAE,RMSE,R2,N_test
0,ARX MI>0.09,0.078398,0.09504,-0.220173,2015


,Variable,Coeficiente estandarizado
29,EMA_6h_model,0.591744
31,EMA_12h_model,-0.547453
34,EMA_4h_model,-0.461451
18,EMA_1d_model,0.417969
22,EMA_3d_model,-0.319637
35,EMA_2h_model,0.218677
27,EMA_2m_model,-0.178169
12,EMA_1m_model,0.136247
23,MACD_1s_2m_model,0.124372
21,MACD_1s_1m_model,-0.103554


## XGBoost


In [ ]:
# XGBoost usa exactamente las mismas variables que el ARX para una comparación justa
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=5,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.05,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(arx_train[arx_features], arx_train["BTC_yo_model"])
pred_xgb = np.clip(xgb_model.predict(arx_test[arx_features]), 0.0, 1.0)

resultado_xgb = evaluar_modelo("XGBoost MI>0.09", arx_test["BTC_yo_model"], pred_xgb)
resultados_modelos.append(resultado_xgb)

importancia_xgb = pd.DataFrame({
    "Variable": arx_features,
    "Importancia": xgb_model.feature_importances_
}).sort_values("Importancia", ascending=False)

display(pd.DataFrame(resultados_modelos).drop_duplicates("Modelo", keep="last"))
display(importancia_xgb.head(50))


,Modelo,MAE,RMSE,R2,N_test
0,Persistencia ingenua,0.067986,0.090405,-0.104054,2015
1,AR,0.058628,0.075802,0.223815,2015
2,ARX MI>0.09,0.078398,0.095040,-0.220173,2015
3,XGBoost MI>0.09,0.060731,0.077945,0.179304,2015


,Variable,Importancia
0,BTC_p_lag0,0.179815
6,BTC_shannon_model,0.178638
9,RLIQ_1h_model,0.071554
7,RLIQ_30_model,0.050416
11,RLIQ_2h_model,0.045546
8,RLIQ_20_model,0.026335
14,EMA_3m_model,0.025002
10,nBTC_model,0.021204
20,RLIQ_6h_model,0.020240
22,EMA_3d_model,0.018604


## Selección temporal y ablación XGBoost


In [ ]:
# Diseño temporal: train de selección | validación | test final intacto
from sklearn.inspection import permutation_importance

VALIDATION_SIZE = 2016
PURGE = HORIZON

posiciones = np.arange(len(model_df))
fin_desarrollo = split_pos - PURGE
inicio_validacion = fin_desarrollo - VALIDATION_SIZE

selection_train_mask = posiciones < inicio_validacion
validation_mask = (posiciones >= inicio_validacion) & (posiciones < fin_desarrollo)
final_train_mask = posiciones < fin_desarrollo

def nuevo_xgb():
    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=5,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.05,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )

# La MI se recalcula sin tocar validación ni test
mi_rows_selection = []
for variable in candidate_features:
    subset = model_df.loc[selection_train_mask, [variable, "BTC_yo_model"]]
    subset = subset.replace([np.inf, -np.inf], np.nan).dropna()
    if len(subset) >= 100 and subset[variable].nunique() >= 2:
        valor = mutual_info_regression(
            subset[[variable]], subset["BTC_yo_model"], random_state=42
        )[0]
        mi_rows_selection.append({"Variable": variable, "MI selección": valor})

mi_selection_temporal = pd.DataFrame(mi_rows_selection).sort_values(
    "MI selección", ascending=False
)
mi_candidates = mi_selection_temporal.loc[
    mi_selection_temporal["MI selección"] > MI_THRESHOLD, "Variable"
].tolist()

if not mi_candidates:
    raise RuntimeError("Ninguna variable superó MI > 0.09 en train de selección.")

selection_features = ar_features + mi_candidates
selection_cols = selection_features + ["BTC_yo_model"]
selection_train = model_df.loc[selection_train_mask, selection_cols].replace(
    [np.inf, -np.inf], np.nan
).dropna()
selection_val = model_df.loc[validation_mask, selection_cols].replace(
    [np.inf, -np.inf], np.nan
).dropna()

xgb_selection_full = nuevo_xgb()
xgb_selection_full.fit(
    selection_train[selection_features], selection_train["BTC_yo_model"]
)
pred_val_full = np.clip(
    xgb_selection_full.predict(selection_val[selection_features]), 0.0, 1.0
)
rmse_val_full = np.sqrt(mean_squared_error(selection_val["BTC_yo_model"], pred_val_full))

perm = permutation_importance(
    xgb_selection_full,
    selection_val[selection_features],
    selection_val["BTC_yo_model"],
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)
permutation_df = pd.DataFrame({
    "Variable": selection_features,
    "Importancia permutación": perm.importances_mean,
    "Desvío": perm.importances_std,
}).sort_values("Importancia permutación", ascending=False)

# Sólo factores exógenos con contribución positiva en validación
positive_factors = permutation_df.loc[
    (permutation_df["Importancia permutación"] > 0)
    & (~permutation_df["Variable"].isin(ar_features)),
    "Variable",
].tolist()

# Nuevo punto de referencia: sólo rezagos + factores con permutación positiva
positive_features = ar_features + positive_factors
positive_cols = positive_features + ["BTC_yo_model"]
positive_train = model_df.loc[selection_train_mask, positive_cols].replace(
    [np.inf, -np.inf], np.nan
).dropna()
positive_val = model_df.loc[validation_mask, positive_cols].replace(
    [np.inf, -np.inf], np.nan
).dropna()
xgb_positive = nuevo_xgb()
xgb_positive.fit(positive_train[positive_features], positive_train["BTC_yo_model"])
pred_val_positive = np.clip(xgb_positive.predict(positive_val[positive_features]), 0.0, 1.0)
rmse_val_positive = np.sqrt(
    mean_squared_error(positive_val["BTC_yo_model"], pred_val_positive)
)

print(f"Train selección: {len(selection_train):,}; validación: {len(selection_val):,}")
print(f"Candidatas MI > {MI_THRESHOLD}: {len(mi_candidates)}")
print(f"Factores con permutación positiva: {len(positive_factors)}")
display(mi_selection_temporal.head(50))
display(permutation_df.head(50))


Train selección: 9,903; validación: 2,016
Candidatas MI > 0.09: 35
Factores con permutación positiva: 16


,Variable,MI selección
0,BTC_shannon_model,0.195800
90,RLIQ_30_model,0.189203
94,RLIQ_1h_model,0.177689
86,RLIQ_20_model,0.176460
15,nBTC_model,0.170061
98,RLIQ_2h_model,0.151526
12,EMA_1m_model,0.129757
14,EMA_3m_model,0.129602
132,RSTAB_2m_model,0.126340
129,RSTAB_1m_model,0.125554


,Variable,Importancia permutación,Desvío
6,BTC_shannon_model,4.278327e-03,0.000237
8,RLIQ_1h_model,3.895291e-03,0.000126
11,RLIQ_2h_model,3.006728e-03,0.000103
1,BTC_p_lag1,3.005230e-03,0.000204
0,BTC_p_lag0,2.537867e-03,0.000243
7,RLIQ_30_model,1.487701e-03,0.000130
9,RLIQ_20_model,1.329889e-03,0.000073
5,BTC_p_lag5,1.167343e-03,0.000108
3,BTC_p_lag3,1.164616e-03,0.000078
26,EMA_6h_model,5.696088e-04,0.000288


### Ablación por familias


In [ ]:
# Ablación: quitar una familia cada vez y medir el cambio de RMSE en validación
def familia(variable):
    if variable == "BTC_shannon_model": return "SHANNON"
    if variable == "nBTC_model": return "nBTC"
    return variable.split("_", 1)[0]

familias = sorted({familia(v) for v in positive_factors})
ablation_rows = []

for grupo in familias:
    removidas = [v for v in positive_factors if familia(v) == grupo]
    features_sin_grupo = ar_features + [v for v in positive_factors if v not in removidas]
    cols = features_sin_grupo + ["BTC_yo_model"]
    train_g = model_df.loc[selection_train_mask, cols].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    val_g = model_df.loc[validation_mask, cols].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    modelo_g = nuevo_xgb()
    modelo_g.fit(train_g[features_sin_grupo], train_g["BTC_yo_model"])
    pred_g = np.clip(modelo_g.predict(val_g[features_sin_grupo]), 0.0, 1.0)
    rmse_g = np.sqrt(mean_squared_error(val_g["BTC_yo_model"], pred_g))
    ablation_rows.append({
        "Familia retirada": grupo,
        "N variables": len(removidas),
        "RMSE sin familia": rmse_g,
        "Delta RMSE": rmse_g - rmse_val_positive,
        "Aporta fuera de muestra": rmse_g > rmse_val_positive,
    })

ablation_df = pd.DataFrame(ablation_rows).sort_values("Delta RMSE", ascending=False)
familias_utiles = ablation_df.loc[
    ablation_df["Aporta fuera de muestra"], "Familia retirada"
].tolist()
selected_clean_factors = [v for v in positive_factors if familia(v) in familias_utiles]

print(f"RMSE tras filtro de permutación en validación: {rmse_val_positive:.6f}")
print(f"Familias retenidas: {familias_utiles}")
print(f"Variables limpias retenidas: {len(selected_clean_factors)}")
display(ablation_df)
display(permutation_df[permutation_df["Variable"].isin(selected_clean_factors)])


RMSE tras filtro de permutación en validación: 0.080638
Familias retenidas: ['nBTC', 'RLIQ', 'SHANNON', 'RSIp']
Variables limpias retenidas: 9


,Familia retirada,N variables,RMSE sin familia,Delta RMSE,Aporta fuera de muestra
5,nBTC,1,0.086215,0.005576,True
1,RLIQ,6,0.085722,0.005083,True
4,SHANNON,1,0.084509,0.003871,True
2,RSIp,1,0.081940,0.001302,True
3,RSTAB,2,0.075041,-0.005597,False
0,EMA,5,0.072372,-0.008266,False


,Variable,Importancia permutación,Desvío
6,BTC_shannon_model,0.004278,0.000237
8,RLIQ_1h_model,0.003895,0.000126
11,RLIQ_2h_model,0.003007,0.000103
7,RLIQ_30_model,0.001488,0.000130
9,RLIQ_20_model,0.001330,0.000073
10,nBTC_model,0.000559,0.000043
16,RLIQ_4h_model,0.000407,0.000060
35,RLIQ_2m_model,0.000010,0.000006
17,RSIp_3d_model,0.000007,0.000021


### Comparación final: XGBoost autoregresivo vs XGBoost limpio


In [ ]:
# Reentrenamiento final sin usar el test para seleccionar variables
comparacion_xgb = []

for nombre, features in {
    "XGBoost AR (sólo rezagos)": ar_features,
    "XGBoost limpio (AR + factores)": ar_features + selected_clean_factors,
}.items():
    cols = features + ["BTC_yo_model"]
    train_final = model_df.loc[final_train_mask, cols].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    test_final = model_df.loc[test_mask, cols].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    modelo_final = nuevo_xgb()
    modelo_final.fit(train_final[features], train_final["BTC_yo_model"])
    pred_final = np.clip(modelo_final.predict(test_final[features]), 0.0, 1.0)
    fila = evaluar_modelo(nombre, test_final["BTC_yo_model"], pred_final)
    fila["N_variables"] = len(features)
    comparacion_xgb.append(fila)

comparacion_xgb_df = pd.DataFrame(comparacion_xgb).sort_values("RMSE")
display(comparacion_xgb_df)

mejora = comparacion_xgb_df.set_index("Modelo")
rmse_ar_xgb = mejora.loc["XGBoost AR (sólo rezagos)", "RMSE"]
rmse_limpio = mejora.loc["XGBoost limpio (AR + factores)", "RMSE"]
print(f"Mejora relativa de RMSE por factores: {(rmse_ar_xgb - rmse_limpio) / rmse_ar_xgb:.2%}")

,Modelo,MAE,RMSE,R2,N_test,N_variables
1,XGBoost limpio (AR + factores),0.058513,0.075873,0.222358,2015,15
0,XGBoost AR (sólo rezagos),0.058757,0.076034,0.219051,2015,6


Mejora relativa de RMSE por factores: 0.21%


## Walk-forward: contraste conjunto


In [73]:
# Siete bloques de dos días (288 barras de 10 minutos), ventana expansiva
WF_BLOCK_SIZE = 288
wf_rows = []
wf_predictions = {"AR lineal": [], "XGBoost AR": [], "XGBoost factores": []}

wf_models = {
    "AR lineal": ar_features,
    "XGBoost AR": ar_features,
    "XGBoost factores": ar_features + selected_clean_factors,
}

fold = 0
for fold_start in range(split_pos, len(model_df), WF_BLOCK_SIZE):
    fold_end = min(fold_start + WF_BLOCK_SIZE, len(model_df))
    train_end = fold_start - HORIZON  # purga: ningún target de train entra al bloque evaluado
    fold += 1

    for nombre, features in wf_models.items():
        cols = features + ["BTC_yo_model"]
        train_fold = model_df.iloc[:train_end][cols].replace(
            [np.inf, -np.inf], np.nan
        ).dropna()
        test_fold = model_df.iloc[fold_start:fold_end][cols].replace(
            [np.inf, -np.inf], np.nan
        ).dropna()

        if test_fold.empty:
            continue

        if nombre == "AR lineal":
            modelo = LinearRegression()
        else:
            modelo = nuevo_xgb()

        modelo.fit(train_fold[features], train_fold["BTC_yo_model"])
        pred = np.clip(modelo.predict(test_fold[features]), 0.0, 1.0)
        real = test_fold["BTC_yo_model"].to_numpy()

        fila = evaluar_modelo(nombre, real, pred)
        fila.update({
            "Fold": fold,
            "Inicio": test_fold.index.min(),
            "Fin": test_fold.index.max(),
            "N_train": len(train_fold),
            "N_variables": len(features),
        })
        wf_rows.append(fila)
        wf_predictions[nombre].append(pd.DataFrame({
            "y": real,
            "pred": pred,
        }, index=test_fold.index))

walk_forward_folds = pd.DataFrame(wf_rows)
display(walk_forward_folds[[
    "Fold", "Inicio", "Fin", "Modelo", "MAE", "RMSE", "R2", "N_test", "N_train", "N_variables"
]])


,Fold,Inicio,Fin,Modelo,MAE,RMSE,R2,N_test,N_train,N_variables
0,1,24048,24335,AR lineal,0.050622,0.066510,0.162754,288,24042,6
1,1,24048,24335,XGBoost AR,0.050581,0.066617,0.160069,288,24042,6
2,1,24048,24335,XGBoost factores,0.050372,0.065782,0.180985,288,15984,15
3,2,24336,24623,AR lineal,0.061575,0.078101,0.096694,288,24330,6
4,2,24336,24623,XGBoost AR,0.061328,0.077786,0.103958,288,24330,6
5,2,24336,24623,XGBoost factores,0.060345,0.077626,0.107647,288,16272,15
6,3,24624,24911,AR lineal,0.056862,0.074847,0.226671,288,24618,6
7,3,24624,24911,XGBoost AR,0.056301,0.074286,0.238227,288,24618,6
8,3,24624,24911,XGBoost factores,0.056628,0.074994,0.223642,288,16560,15
9,4,24912,25199,AR lineal,0.058505,0.075310,0.342880,288,24906,6


### Resultado agregado y estabilidad por bloques


In [74]:
# Métricas agregadas sobre todas las predicciones estrictamente fuera de muestra
aggregate_rows = []
for nombre, partes in wf_predictions.items():
    pred_df = pd.concat(partes).sort_index()
    fila = evaluar_modelo(nombre, pred_df["y"], pred_df["pred"])
    fila["Folds ganados por RMSE"] = int(
        (walk_forward_folds.loc[
            walk_forward_folds.groupby("Fold")["RMSE"].idxmin(), "Modelo"
        ] == nombre).sum()
    )
    fila["RMSE medio por fold"] = walk_forward_folds.loc[
        walk_forward_folds["Modelo"] == nombre, "RMSE"
    ].mean()
    fila["Desvío RMSE"] = walk_forward_folds.loc[
        walk_forward_folds["Modelo"] == nombre, "RMSE"
    ].std()
    aggregate_rows.append(fila)

walk_forward_resumen = pd.DataFrame(aggregate_rows).sort_values("RMSE")
display(walk_forward_resumen)

rmse_xgb_ar = walk_forward_resumen.set_index("Modelo").loc["XGBoost AR", "RMSE"]
rmse_xgb_factores = walk_forward_resumen.set_index("Modelo").loc["XGBoost factores", "RMSE"]
mejora_factores_wf = (rmse_xgb_ar - rmse_xgb_factores) / rmse_xgb_ar
print(f"Mejora walk-forward de factores frente a XGBoost AR: {mejora_factores_wf:.2%}")

# Tabla directa de diferencias por bloque: positivo significa que los factores mejoran
comparacion_folds = walk_forward_folds.pivot(index="Fold", columns="Modelo", values="RMSE")
comparacion_folds["Mejora factores vs XGB AR"] = (
    comparacion_folds["XGBoost AR"] - comparacion_folds["XGBoost factores"]
)
comparacion_folds["Mejora factores vs XGB AR (%)"] = (
    comparacion_folds["Mejora factores vs XGB AR"] / comparacion_folds["XGBoost AR"]
)
display(comparacion_folds)

,Modelo,MAE,RMSE,R2,N_test,Folds ganados por RMSE,RMSE medio por fold,Desvío RMSE
2,XGBoost factores,0.058535,0.075777,0.224313,2015,5,0.075596,0.005662
0,AR lineal,0.058612,0.075787,0.224112,2015,1,0.075623,0.005401
1,XGBoost AR,0.058630,0.075921,0.221359,2015,1,0.075756,0.005426


Mejora walk-forward de factores frente a XGBoost AR: 0.19%


Modelo,AR lineal,XGBoost AR,XGBoost factores,Mejora factores vs XGB AR,Mejora factores vs XGB AR (%)
Fold,,,,,
1,0.066510,0.066617,0.065782,0.000835,0.012530
2,0.078101,0.077786,0.077626,0.000160,0.002060
3,0.074847,0.074286,0.074994,-0.000708,-0.009527
4,0.075310,0.075936,0.078715,-0.002779,-0.036598
5,0.072603,0.072913,0.071401,0.001512,0.020738
6,0.084022,0.084161,0.083444,0.000718,0.008527
7,0.077967,0.078596,0.077214,0.001382,0.017583


In [1]:
# XGBoost sobre factores obtiene el menor RMSE (0.07578) y gana 5/7 folds.
# Sin embargo, su ventaja frente al AR lineal es marginal (<0.02% en RMSE).
# El XGBoost-AR no mejora al AR lineal, sugiriendo que la estructura
# autorregresiva capturada es principalmente lineal.
# R² ≈ 0.22 en los tres modelos: capacidad predictiva moderada y muy similar.
# RESULTADO: los factores contienen señal predictiva real y consistente,
# pero todavía no muestran una mejora material sobre la persistencia AR.